<a href="https://colab.research.google.com/github/daldowaihi/Beyond-Accuracy-Explanation-Instability-Under-Decision-Uncertainty-in-Credit-Risk-Classification/blob/main/Credit_XAI_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond Accuracy: Explanation Instability Under Decision Uncertainty in Credit Risk Classification



| Component | Choice |
|---|---|
| **Models** | Random Forest (bagging) + Gradient Boosting (boosting) |
| **Explainers** | TreeSHAP + LIME |
| **Strata** | Confident-Correct · Borderline · Confident-Incorrect |
| **Experiments** | A: Perturbation stability · B: Cross-method agreement |
| **Headline metrics** | Spearman ρ + Top-5 Jaccard |
| **Statistical test** | Mann–Whitney U + Holm–Bonferroni + rank-biserial |
| **Reproducibility seed** | `random_state = 42` |

**LocPI**, the local permutation-importance method, is computed during the run as a robustness check and reported in the Appendix only — it is *not* part of the headline analysis.

## How to run on Colab

1. **Runtime → Run all** (or run cells one at a time, top to bottom).
2. All outputs are written to `./outputs/{tables,figures,models}/` and zipped at the end. The file `outputs/tables/RESULTS_HEADLINE.md` contains the consolidated numbers needed to write the Results section.


## 1 · Environment setup

In [1]:
# Colab already has numpy, pandas, sklearn, matplotlib, seaborn, scipy.
# We need shap, lime, lightgbm.
!pip -q install shap lime lightgbm


## 2 · Imports and reproducibility

In [2]:
import argparse, json, logging, os, random, warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from scipy import stats as sp_stats

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("optA")

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)


## 3 · Configuration

All experimental parameters from the methodology are gathered here for one-stop visibility. Switch `QUICK_MODE` to `True` for a fast smoke-test run.

In [3]:
# ---- Quick-mode flag ------------------------------------------------------
QUICK_MODE = True

# ---- Data split -------------------------------------------
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# ---- Stratum bands --------------------------------------
CONFIDENT_LOW, CONFIDENT_HIGH = 0.20, 0.80
BORDERLINE_LOW, BORDERLINE_HIGH = 0.40, 0.60

# ---- Perturbation protocol ------------------------------
N_PERTURBATIONS = 30
PERTURB_SIGMA_HEADLINE = 0.03
PERTURB_SIGMA_GRID = (0.01, 0.03, 0.05)
ORDINAL_SHIFT_PROB = 0.20
PRED_STABILITY_FILTER = 0.05  # |Δp̂| < 0.05

# ---- Sampling -------------------------------------------------------------
INSTANCES_PER_STRATUM = 100

# ---- LIME -----------------------------------------------------------------
LIME_NUM_SAMPLES = 5000

# ---- Statistical tests (Section III-H) ------------------------------------
REFERENCE_STRATUM = "confident_correct"
ALPHA = 0.05

# ---- Apply quick mode -----------------------------------------------------
if QUICK_MODE:
    INSTANCES_PER_STRATUM = 20
    N_PERTURBATIONS = 10
    LIME_NUM_SAMPLES = 1000
    PERTURB_SIGMA_GRID = (0.03,)
    print("[QUICK MODE ON]")

# ---- Output paths ---------------------------------------------------------
OUTPUT_DIR = Path("outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
for d in (TABLES_DIR, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"INSTANCES_PER_STRATUM = {INSTANCES_PER_STRATUM}")
print(f"N_PERTURBATIONS       = {N_PERTURBATIONS}")
print(f"PERTURB_SIGMA_GRID    = {PERTURB_SIGMA_GRID}")
print(f"OUTPUT_DIR            = {OUTPUT_DIR.resolve()}")


INSTANCES_PER_STRATUM = 100
N_PERTURBATIONS       = 30
PERTURB_SIGMA_GRID    = (0.01, 0.03, 0.05)
OUTPUT_DIR            = /content/outputs


## 4 · Data loading and 70/15/15 split

Loads the UCI *Default of Credit Card Clients* dataset (Yeh & Lien, 2009) and applies the descriptive column-rename mapping documented in Appendix A. Splits 70/15/15 stratified by the target.

In [4]:
COLUMN_RENAME = {
    "LIMIT_BAL": "Credit_Limit",
    "SEX": "Gender",
    "EDUCATION": "Education_Level",
    "MARRIAGE": "Marital_Status",
    "AGE": "Age",
    "PAY_0": "RepayDelay_Sep", "PAY_2": "RepayDelay_Aug",
    "PAY_3": "RepayDelay_Jul", "PAY_4": "RepayDelay_Jun",
    "PAY_5": "RepayDelay_May", "PAY_6": "RepayDelay_Apr",
    "BILL_AMT1": "BillAmt_Sep", "BILL_AMT2": "BillAmt_Aug",
    "BILL_AMT3": "BillAmt_Jul", "BILL_AMT4": "BillAmt_Jun",
    "BILL_AMT5": "BillAmt_May", "BILL_AMT6": "BillAmt_Apr",
    "PAY_AMT1": "Payment_Sep", "PAY_AMT2": "Payment_Aug",
    "PAY_AMT3": "Payment_Jul", "PAY_AMT4": "Payment_Jun",
    "PAY_AMT5": "Payment_May", "PAY_AMT6": "Payment_Apr",
}

CONTINUOUS_FEATURES = [
    "Credit_Limit", "Age",
    "BillAmt_Sep", "BillAmt_Aug", "BillAmt_Jul",
    "BillAmt_Jun", "BillAmt_May", "BillAmt_Apr",
    "Payment_Sep", "Payment_Aug", "Payment_Jul",
    "Payment_Jun", "Payment_May", "Payment_Apr",
]
ORDINAL_FEATURES = [
    "RepayDelay_Sep", "RepayDelay_Aug", "RepayDelay_Jul",
    "RepayDelay_Jun", "RepayDelay_May", "RepayDelay_Apr",
]
CATEGORICAL_FEATURES = ["Gender", "Education_Level", "Marital_Status"]
ALL_FEATURES = CONTINUOUS_FEATURES + ORDINAL_FEATURES + CATEGORICAL_FEATURES


def load_uci_dataset() -> pd.DataFrame:
    # Load the UCI dataset, applying COLUMN_RENAME and folding undocumented levels.
    import urllib.request
    p = Path("data/default_of_credit_card_clients.xls")
    p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists():
        url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/00350/"
               "default%20of%20credit%20card%20clients.xls")
        log.info(f"Downloading UCI dataset → {p}")
        urllib.request.urlretrieve(url, p)
    df = pd.read_excel(p, header=1)
    df = df.rename(columns={"default payment next month": "default", **COLUMN_RENAME})
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])
    df.loc[df["Education_Level"].isin([0, 5, 6]), "Education_Level"] = 4
    df.loc[df["Marital_Status"] == 0, "Marital_Status"] = 3
    log.info(f"Loaded {len(df)} rows, default rate={df['default'].mean():.3f}")
    return df


def split_data(df: pd.DataFrame, seed: int = SEED):
    X = df[ALL_FEATURES].copy()
    y = df["default"].astype(int).values
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=TEST_FRAC, stratify=y, random_state=seed
    )
    val_share = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_share,
        stratify=y_trainval, random_state=seed,
    )
    log.info(
        f"Train {len(X_train)} (def={y_train.mean():.3f}), "
        f"Val {len(X_val)} (def={y_val.mean():.3f}), "
        f"Test {len(X_test)} (def={y_test.mean():.3f})"
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


df = load_uci_dataset()
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)


## 5 · Train Random Forest and Gradient Boosting

Both models share the same train/val/test split and use class-balanced weighting. A small validation grid is searched and the configuration with the highest validation AUC-ROC is kept.

In [5]:
def train_random_forest(X_train, y_train, X_val, y_val, seed=SEED):
    grid = []
    for n_est in [500]:
        for max_d in [None, 10, 20]:
            for min_leaf in [1, 5, 10]:
                model = RandomForestClassifier(
                    n_estimators=n_est, max_depth=max_d, min_samples_leaf=min_leaf,
                    class_weight="balanced", n_jobs=-1, random_state=seed,
                )
                model.fit(X_train, y_train)
                p_val = model.predict_proba(X_val)[:, 1]
                auc = roc_auc_score(y_val, p_val)
                grid.append({
                    "model": "random_forest", "n_estimators": n_est,
                    "max_depth": max_d, "min_samples_leaf": min_leaf,
                    "val_auc_roc": auc,
                })
                log.info(f"  RF n={n_est} d={max_d} leaf={min_leaf} → val_auc={auc:.4f}")
    grid_df = pd.DataFrame(grid).sort_values("val_auc_roc", ascending=False).reset_index(drop=True)
    best = grid_df.iloc[0]
    final = RandomForestClassifier(
        n_estimators=int(best["n_estimators"]),
        max_depth=None if pd.isna(best["max_depth"]) else int(best["max_depth"]),
        min_samples_leaf=int(best["min_samples_leaf"]),
        class_weight="balanced", n_jobs=-1, random_state=seed,
    )
    final.fit(X_train, y_train)
    log.info(f"RF best: {dict(best)}")
    return final, grid_df


def train_gradient_boosting(X_train, y_train, X_val, y_val, seed=SEED):
    try:
        from lightgbm import LGBMClassifier
        backend = "lightgbm"
    except ImportError:
        LGBMClassifier = None
        backend = "histgbm"
    log.info(f"GBM backend: {backend}")
    grid = []
    for n_est in [200, 500]:
        for max_d in [4, 6, 8]:
            for lr in [0.05, 0.10]:
                if backend == "lightgbm":
                    model = LGBMClassifier(
                        n_estimators=n_est, max_depth=max_d, learning_rate=lr,
                        class_weight="balanced", n_jobs=-1, random_state=seed,
                        verbose=-1,
                    )
                else:
                    model = HistGradientBoostingClassifier(
                        max_iter=n_est, max_depth=max_d, learning_rate=lr,
                        class_weight="balanced", random_state=seed,
                    )
                model.fit(X_train, y_train)
                p_val = model.predict_proba(X_val)[:, 1]
                auc = roc_auc_score(y_val, p_val)
                grid.append({
                    "model": "gradient_boosting", "n_estimators": n_est,
                    "max_depth": max_d, "learning_rate": lr,
                    "val_auc_roc": auc, "backend": backend,
                })
                log.info(f"  GBM n={n_est} d={max_d} lr={lr} → val_auc={auc:.4f}")
    grid_df = pd.DataFrame(grid).sort_values("val_auc_roc", ascending=False).reset_index(drop=True)
    best = grid_df.iloc[0]
    if backend == "lightgbm":
        from lightgbm import LGBMClassifier
        final = LGBMClassifier(
            n_estimators=int(best["n_estimators"]), max_depth=int(best["max_depth"]),
            learning_rate=float(best["learning_rate"]),
            class_weight="balanced", n_jobs=-1, random_state=seed, verbose=-1,
        )
    else:
        final = HistGradientBoostingClassifier(
            max_iter=int(best["n_estimators"]), max_depth=int(best["max_depth"]),
            learning_rate=float(best["learning_rate"]),
            class_weight="balanced", random_state=seed,
        )
    final.fit(X_train, y_train)
    log.info(f"GBM best: {dict(best)}")
    return final, grid_df


rf, rf_grid = train_random_forest(X_train, y_train, X_val, y_val)
gbm, gbm_grid = train_gradient_boosting(X_train, y_train, X_val, y_val)
rf_grid.to_csv(TABLES_DIR / "rf_hp_grid.csv", index=False)
gbm_grid.to_csv(TABLES_DIR / "gbm_hp_grid.csv", index=False)
print("Best RF:", dict(rf_grid.iloc[0]))
print("Best GBM:", dict(gbm_grid.iloc[0]))


Best RF: {'model': 'random_forest', 'n_estimators': np.int64(500), 'max_depth': np.float64(nan), 'min_samples_leaf': np.int64(10), 'val_auc_roc': np.float64(0.7830540720722012)}
Best GBM: {'model': 'gradient_boosting', 'n_estimators': np.int64(200), 'max_depth': np.int64(4), 'learning_rate': np.float64(0.05), 'val_auc_roc': np.float64(0.7816676822056072), 'backend': 'lightgbm'}


## 6 · Test-set performance

Computes the metrics.

In [6]:
def evaluate_model(model, X_test, y_test, threshold=0.5):
    p_test = model.predict_proba(X_test)[:, 1]
    y_pred = (p_test >= threshold).astype(int)
    return {
        "auc_roc": roc_auc_score(y_test, p_test),
        "auc_pr": average_precision_score(y_test, p_test),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_default": precision_score(y_test, y_pred),
        "recall_default": recall_score(y_test, y_pred),
        "f1_default": f1_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    }


rf_metrics = evaluate_model(rf, X_test, y_test)
gbm_metrics = evaluate_model(gbm, X_test, y_test)
test_metrics = pd.DataFrame({"random_forest": rf_metrics, "gradient_boosting": gbm_metrics})
test_metrics.to_csv(TABLES_DIR / "test_metrics.csv")
print(test_metrics.round(4).to_string())

# Confusion matrices
rf_cm = confusion_matrix(y_test, rf.predict(X_test))
gbm_cm = confusion_matrix(y_test, gbm.predict(X_test))
pd.DataFrame(rf_cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(
    TABLES_DIR / "confusion_random_forest.csv")
pd.DataFrame(gbm_cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(
    TABLES_DIR / "confusion_gradient_boosting.csv")
print("\nRF confusion:\n", rf_cm)
print("\nGBM confusion:\n", gbm_cm)


                   random_forest  gradient_boosting
auc_roc                   0.7787             0.7778
auc_pr                    0.5609             0.5557
accuracy                  0.7927             0.7649
precision_default         0.5301             0.4754
recall_default            0.5487             0.6131
f1_default                0.5393             0.5356
balanced_accuracy         0.7053             0.7105

RF confusion:
 [[3021  484]
 [ 449  546]]

GBM confusion:
 [[2832  673]
 [ 385  610]]


## 7 · Stratify test predictions (3 strata — Option A)

Operational definitions:

| Stratum | Confidence band | Correctness |
|---|---|---|
| Confident–Correct | p̂ ≤ 0.20 or p̂ ≥ 0.80 | Predicted class equals true label |
| Borderline | 0.40 ≤ p̂ ≤ 0.60 | Either label outcome |
| Confident–Incorrect | p̂ ≤ 0.20 or p̂ ≥ 0.80 | Predicted class differs from true label |

Predictions in the gap regions (0.20, 0.40) ∪ (0.60, 0.80) are excluded from the headline analysis.

In [7]:
def stratify_test_set(model, X_test, y_test) -> pd.DataFrame:
    p_test = model.predict_proba(X_test)[:, 1]
    pred = (p_test >= 0.5).astype(int)
    correct = (pred == y_test)
    confident = (p_test <= CONFIDENT_LOW) | (p_test >= CONFIDENT_HIGH)
    borderline = (p_test >= BORDERLINE_LOW) & (p_test <= BORDERLINE_HIGH)
    stratum = np.full(len(y_test), "other", dtype=object)
    stratum[confident & correct] = "confident_correct"
    stratum[confident & (~correct)] = "confident_incorrect"
    stratum[borderline] = "borderline"
    return pd.DataFrame({
        "test_idx": np.arange(len(y_test)),
        "p_default": p_test, "predicted": pred, "true": y_test,
        "correct": correct, "stratum": stratum,
    })


def sample_per_stratum(strata_df, n_per_stratum=INSTANCES_PER_STRATUM, seed=SEED):
    rng = np.random.RandomState(seed)
    out = {}
    for s in ["confident_correct", "borderline", "confident_incorrect"]:
        idx = strata_df.index[strata_df["stratum"] == s].tolist()
        if len(idx) > n_per_stratum:
            idx = rng.choice(idx, size=n_per_stratum, replace=False).tolist()
        out[s] = sorted(idx)
    return out


strata_rf = stratify_test_set(rf, X_test, y_test)
strata_gbm = stratify_test_set(gbm, X_test, y_test)
counts = pd.DataFrame({
    "random_forest": strata_rf["stratum"].value_counts(),
    "gradient_boosting": strata_gbm["stratum"].value_counts(),
}).fillna(0).astype(int)
counts = counts.reindex(["confident_correct", "borderline", "confident_incorrect", "other"])
counts.to_csv(TABLES_DIR / "stratum_counts.csv")
print("Stratum counts (test set):")
print(counts)

sample_rf = sample_per_stratum(strata_rf)
sample_gbm = sample_per_stratum(strata_gbm)
print("\nSampled per stratum (RF):", {k: len(v) for k, v in sample_rf.items()})
print("Sampled per stratum (GBM):", {k: len(v) for k, v in sample_gbm.items()})


Stratum counts (test set):
                     random_forest  gradient_boosting
stratum                                              
confident_correct             1341                985
borderline                     575                971
confident_incorrect            145                173
other                         2439               2371

Sampled per stratum (RF): {'confident_correct': 100, 'borderline': 100, 'confident_incorrect': 100}
Sampled per stratum (GBM): {'confident_correct': 100, 'borderline': 100, 'confident_incorrect': 100}


## 8 · Stability and agreement metrics

Two headline metrics.

In [8]:
def spearman_rho(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    rho, _ = sp_stats.spearmanr(a, b)
    return float(rho) if not np.isnan(rho) else np.nan


def top_k_jaccard(a, b, k=5):
    a, b = np.abs(np.asarray(a)), np.abs(np.asarray(b))
    top_a = set(np.argsort(a)[-k:].tolist())
    top_b = set(np.argsort(b)[-k:].tolist())
    union = top_a | top_b
    return len(top_a & top_b) / len(union) if union else np.nan


## 9 · Type-aware perturber

Continuous: additive Gaussian σ × std(j). Ordinal: ±1 shift with probability 0.20, clipped to training range. Categorical: held fixed.

In [9]:
class TypeAwarePerturber:
    def __init__(self, X_train, sigma=PERTURB_SIGMA_HEADLINE,
                 ordinal_shift_prob=ORDINAL_SHIFT_PROB, seed=SEED):
        self.sigma = sigma
        self.ordinal_shift_prob = ordinal_shift_prob
        self.feat_names = list(X_train.columns)
        self.continuous = [f for f in CONTINUOUS_FEATURES if f in self.feat_names]
        self.ordinal = [f for f in ORDINAL_FEATURES if f in self.feat_names]
        self.cont_std = X_train[self.continuous].std().to_dict()
        self.ord_min = X_train[self.ordinal].min().to_dict()
        self.ord_max = X_train[self.ordinal].max().to_dict()
        self.rng = np.random.RandomState(seed)

    def perturb(self, x: pd.Series, n: int) -> pd.DataFrame:
        rows = []
        for _ in range(n):
            xp = x.astype(float).copy()
            for f in self.continuous:
                xp[f] = xp[f] + self.rng.normal(0, self.sigma * self.cont_std[f])
            for f in self.ordinal:
                if self.rng.rand() < self.ordinal_shift_prob:
                    shift = self.rng.choice([-1, 1])
                    xp[f] = float(np.clip(xp[f] + shift, self.ord_min[f], self.ord_max[f]))
            rows.append(xp)
        return pd.DataFrame(rows).reset_index(drop=True)


## 10 · Explainers — TreeSHAP and LIME

Build SHAP and LIME explainers per model. Each per-instance attribution is a 1-D vector of length p (one weight per feature).

In [10]:
def build_explainers(model, X_train):
    import shap
    from lime.lime_tabular import LimeTabularExplainer
    try:
        shap_expl = shap.TreeExplainer(model)
    except Exception as e:
        log.warning(f"TreeExplainer init failed ({e}); using generic Explainer.")
        shap_expl = shap.Explainer(model.predict_proba, X_train.iloc[:200])
    feat_names = list(X_train.columns)
    cat_idx = [feat_names.index(f) for f in CATEGORICAL_FEATURES if f in feat_names]
    lime_expl = LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=feat_names,
        class_names=["non_default", "default"],
        categorical_features=cat_idx,
        discretize_continuous=False,
        random_state=SEED,
    )
    return shap_expl, lime_expl


def shap_attribution(shap_expl, x_row: pd.Series) -> np.ndarray:
    arr = np.asarray(x_row.values).reshape(1, -1)
    sv = shap_expl(arr) if callable(shap_expl) else shap_expl.shap_values(arr)
    if hasattr(sv, "values"):
        v = sv.values
        if v.ndim == 3:
            v = v[0, :, 1]
        elif v.ndim == 2:
            v = v[0]
        else:
            v = v.flatten()
    elif isinstance(sv, list):
        v = np.asarray(sv[1]).flatten() if len(sv) > 1 else np.asarray(sv[0]).flatten()
    else:
        v = np.asarray(sv).flatten()
    return v


def lime_attribution(lime_expl, model, x_row: pd.Series, n_features: int) -> np.ndarray:
    expl = lime_expl.explain_instance(
        data_row=x_row.values,
        predict_fn=model.predict_proba,
        num_features=n_features,
        num_samples=LIME_NUM_SAMPLES,
        labels=[1],
    )
    weights = np.zeros(n_features)
    for fid, w in expl.as_map()[1]:
        weights[fid] = w
    return weights


## 11 · Experiment A — Input perturbation stability

For each instance:
1. Generate 30 type-aware neighbors at σ = 0.03.
2. Keep only neighbors with **|Δp̂| < 0.05** (the prediction-stability filter — this isolates explanation change from prediction change).
3. Compute SHAP and LIME attributions on the original instance and each surviving neighbor.
4. Report the per-instance median Spearman ρ and median Top-5 Jaccard for both methods.

In [11]:
def run_experiment_A(model, model_name, X_test, X_train, sample, perturber,
                     pred_filter=PRED_STABILITY_FILTER, sigma_label="0.03"):
    shap_expl, lime_expl = build_explainers(model, X_train)
    rows = []
    n_feat = X_test.shape[1]
    for stratum, idx_list in sample.items():
        log.info(f"[Exp A · {model_name} · σ={sigma_label}] {stratum}: {len(idx_list)} instances")
        for idx in idx_list:
            x = X_test.iloc[idx]
            p_orig = model.predict_proba(x.values.reshape(1, -1))[0, 1]
            phi_shap_0 = shap_attribution(shap_expl, x)
            phi_lime_0 = lime_attribution(lime_expl, model, x, n_feat)
            neighbors = perturber.perturb(x, N_PERTURBATIONS)
            p_neigh = model.predict_proba(neighbors.values)[:, 1]
            keep = np.abs(p_neigh - p_orig) < pred_filter
            kept = neighbors.loc[keep].reset_index(drop=True)
            shap_rhos, shap_jaccs, lime_rhos, lime_jaccs = [], [], [], []
            for _, n_row in kept.iterrows():
                phi_shap_n = shap_attribution(shap_expl, n_row)
                phi_lime_n = lime_attribution(lime_expl, model, n_row, n_feat)
                shap_rhos.append(spearman_rho(phi_shap_0, phi_shap_n))
                shap_jaccs.append(top_k_jaccard(phi_shap_0, phi_shap_n, k=5))
                lime_rhos.append(spearman_rho(phi_lime_0, phi_lime_n))
                lime_jaccs.append(top_k_jaccard(phi_lime_0, phi_lime_n, k=5))
            rows.append({
                "model": model_name, "stratum": stratum, "test_idx": idx,
                "p_orig": float(p_orig), "n_kept": int(keep.sum()),
                "shap_med_rho":  float(np.nanmedian(shap_rhos))  if shap_rhos else np.nan,
                "shap_med_jacc": float(np.nanmedian(shap_jaccs)) if shap_jaccs else np.nan,
                "lime_med_rho":  float(np.nanmedian(lime_rhos))  if lime_rhos else np.nan,
                "lime_med_jacc": float(np.nanmedian(lime_jaccs)) if lime_jaccs else np.nan,
            })
    return pd.DataFrame(rows)


# Headline σ = 0.03
perturber = TypeAwarePerturber(X_train, sigma=PERTURB_SIGMA_HEADLINE)
expA_rf  = run_experiment_A(rf,  "random_forest",     X_test, X_train, sample_rf,  perturber)
expA_gbm = run_experiment_A(gbm, "gradient_boosting", X_test, X_train, sample_gbm, perturber)
expA_rf.to_csv(TABLES_DIR / "expA_random_forest_per_instance.csv", index=False)
expA_gbm.to_csv(TABLES_DIR / "expA_gradient_boosting_per_instance.csv", index=False)
print("\nRF medians by stratum:")
print(expA_rf.groupby("stratum")[["shap_med_rho","shap_med_jacc","lime_med_rho","lime_med_jacc"]].median().round(3))
print("\nGBM medians by stratum:")
print(expA_gbm.groupby("stratum")[["shap_med_rho","shap_med_jacc","lime_med_rho","lime_med_jacc"]].median().round(3))



RF medians by stratum:
                     shap_med_rho  shap_med_jacc  lime_med_rho  lime_med_jacc
stratum                                                                      
borderline                  0.918          0.667         0.971            1.0
confident_correct           0.962          1.000         0.971            1.0
confident_incorrect         0.953          1.000         0.969            1.0

GBM medians by stratum:
                     shap_med_rho  shap_med_jacc  lime_med_rho  lime_med_jacc
stratum                                                                      
borderline                  0.884          0.667         0.979            1.0
confident_correct           0.955          0.667         0.978            1.0
confident_incorrect         0.948          0.667         0.979            1.0


### Figure A — Per-instance stability distributions by stratum

In [12]:
def figure_expA_distributions(per_instance_df, model_name, sigma_label="0.03"):
    sns.set_style("whitegrid")
    order = ["confident_correct", "borderline", "confident_incorrect"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    for ax, (col, title) in zip(axes.flat, [
        ("shap_med_rho",   "SHAP · Spearman ρ"),
        ("shap_med_jacc",  "SHAP · Top-5 Jaccard"),
        ("lime_med_rho",   "LIME · Spearman ρ"),
        ("lime_med_jacc",  "LIME · Top-5 Jaccard"),
    ]):
        sns.boxplot(data=per_instance_df, x="stratum", y=col, order=order, ax=ax,
                    color="#A8C5E0")
        sns.stripplot(data=per_instance_df, x="stratum", y=col, order=order, ax=ax,
                      color="#1f4e79", size=2.5, jitter=True, alpha=0.5)
        ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel(col)
    fig.suptitle(f"Experiment A — {model_name} (σ={sigma_label})", y=1.00, fontsize=13)
    fig.tight_layout()
    out = FIGURES_DIR / f"expA_{model_name}_sigma{sigma_label}.png"
    fig.savefig(out, dpi=140, bbox_inches="tight"); plt.close(fig)
    print(f"Wrote {out}")


figure_expA_distributions(expA_rf, "random_forest")
figure_expA_distributions(expA_gbm, "gradient_boosting")


Wrote outputs/figures/expA_random_forest_sigma0.03.png
Wrote outputs/figures/expA_gradient_boosting_sigma0.03.png


### σ-sweep (RF only) — robustness check

Methodology Appendix C: repeat Experiment A at σ ∈ {0.01, 0.05} in addition to the headline σ = 0.03 to confirm the qualitative pattern is not an artifact of the perturbation magnitude.

In [13]:
sigma_summary = []
for sigma in PERTURB_SIGMA_GRID:
    if sigma == PERTURB_SIGMA_HEADLINE:
        df_sigma = expA_rf
    else:
        pert = TypeAwarePerturber(X_train, sigma=sigma)
        df_sigma = run_experiment_A(rf, "random_forest", X_test, X_train, sample_rf, pert,
                                    sigma_label=f"{sigma}")
        df_sigma.to_csv(TABLES_DIR / f"expA_random_forest_sigma{sigma}_per_instance.csv",
                        index=False)
        figure_expA_distributions(df_sigma, "random_forest", sigma_label=f"{sigma}")
    med = (df_sigma.groupby("stratum")[["shap_med_rho","shap_med_jacc"]]
           .median().reset_index())
    med["sigma"] = sigma
    sigma_summary.append(med)

sigma_df = pd.concat(sigma_summary, ignore_index=True)
sigma_df.to_csv(TABLES_DIR / "sigma_sweep_summary.csv", index=False)
print(sigma_df.round(3))


Wrote outputs/figures/expA_random_forest_sigma0.01.png
Wrote outputs/figures/expA_random_forest_sigma0.05.png
               stratum  shap_med_rho  shap_med_jacc  sigma
0           borderline         0.969          0.667   0.01
1    confident_correct         0.982          1.000   0.01
2  confident_incorrect         0.978          1.000   0.01
3           borderline         0.918          0.667   0.03
4    confident_correct         0.962          1.000   0.03
5  confident_incorrect         0.953          1.000   0.03
6           borderline         0.868          0.667   0.05
7    confident_correct         0.939          1.000   0.05
8  confident_incorrect         0.926          1.000   0.05


## 12 · Experiment B — Cross-method agreement (SHAP vs LIME)

For each instance, compute SHAP and LIME attributions on the original (unperturbed) input. Report the pairwise Spearman ρ and Top-5 Jaccard between the two attribution vectors, summarised per stratum.

In [14]:
def run_experiment_B(model, model_name, X_test, X_train, sample):
    shap_expl, lime_expl = build_explainers(model, X_train)
    rows = []
    n_feat = X_test.shape[1]
    for stratum, idx_list in sample.items():
        log.info(f"[Exp B · {model_name}] {stratum}: {len(idx_list)} instances")
        for idx in idx_list:
            x = X_test.iloc[idx]
            phi_shap = shap_attribution(shap_expl, x)
            phi_lime = lime_attribution(lime_expl, model, x, n_feat)
            rows.append({
                "model": model_name, "stratum": stratum, "test_idx": idx,
                "shap_lime_rho":  spearman_rho(phi_shap, phi_lime),
                "shap_lime_jacc": top_k_jaccard(phi_shap, phi_lime, k=5),
            })
    return pd.DataFrame(rows)


expB_rf  = run_experiment_B(rf,  "random_forest",     X_test, X_train, sample_rf)
expB_gbm = run_experiment_B(gbm, "gradient_boosting", X_test, X_train, sample_gbm)
expB_rf.to_csv(TABLES_DIR / "expB_random_forest_per_instance.csv", index=False)
expB_gbm.to_csv(TABLES_DIR / "expB_gradient_boosting_per_instance.csv", index=False)

print("\nRF SHAP-vs-LIME medians:")
print(expB_rf.groupby("stratum")[["shap_lime_rho","shap_lime_jacc"]].median().round(3))
print("\nGBM SHAP-vs-LIME medians:")
print(expB_gbm.groupby("stratum")[["shap_lime_rho","shap_lime_jacc"]].median().round(3))



RF SHAP-vs-LIME medians:
                     shap_lime_rho  shap_lime_jacc
stratum                                           
borderline                  -0.431           0.429
confident_correct           -0.029           0.429
confident_incorrect          0.099           0.429

GBM SHAP-vs-LIME medians:
                     shap_lime_rho  shap_lime_jacc
stratum                                           
borderline                  -0.398           0.429
confident_correct            0.166           0.429
confident_incorrect          0.243           0.250


### Figure B — Cross-method agreement heatmap

In [15]:
def figure_expB_heatmap(per_instance_df, model_name):
    order = ["confident_correct", "borderline", "confident_incorrect"]
    pivot = (per_instance_df.groupby("stratum")[["shap_lime_rho", "shap_lime_jacc"]]
             .median().reindex(order))
    fig, ax = plt.subplots(figsize=(6.5, 3.0))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu",
                vmin=0.0, vmax=1.0, ax=ax, cbar_kws={"label": "median agreement"})
    ax.set_title(f"Experiment B — SHAP vs LIME, {model_name}")
    ax.set_ylabel("Stratum")
    fig.tight_layout()
    out = FIGURES_DIR / f"expB_heatmap_{model_name}.png"
    fig.savefig(out, dpi=140, bbox_inches="tight"); plt.close(fig)
    print(f"Wrote {out}")


figure_expB_heatmap(expB_rf, "random_forest")
figure_expB_heatmap(expB_gbm, "gradient_boosting")


Wrote outputs/figures/expB_heatmap_random_forest.png
Wrote outputs/figures/expB_heatmap_gradient_boosting.png


## 13 · Statistical tests (Mann–Whitney U + Holm–Bonferroni)

Each non-reference stratum is compared to Confident-Correct (the reference) using the Mann–Whitney U test, with Holm–Bonferroni correction across the two pairwise comparisons. Rank-biserial effect size is reported alongside.

In [16]:
def rank_biserial(group1, group2):
    g1 = np.asarray(group1, dtype=float); g1 = g1[~np.isnan(g1)]
    g2 = np.asarray(group2, dtype=float); g2 = g2[~np.isnan(g2)]
    if len(g1) == 0 or len(g2) == 0:
        return np.nan
    u, _ = sp_stats.mannwhitneyu(g1, g2, alternative="two-sided")
    return 1.0 - (2.0 * u) / (len(g1) * len(g2))


def holm_bonferroni(p_values):
    p = np.asarray(p_values, dtype=float)
    m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        adj_p = min(p[i] * (m - rank), 1.0)
        running = max(running, adj_p)
        adj[i] = running
    return adj


def stratum_tests(per_instance_df, metric_col, ref=REFERENCE_STRATUM,
                  alternatives=("borderline", "confident_incorrect")):
    ref_vals = per_instance_df.loc[per_instance_df["stratum"] == ref, metric_col].values
    rows = []
    for alt in alternatives:
        alt_vals = per_instance_df.loc[per_instance_df["stratum"] == alt, metric_col].values
        if len(ref_vals) > 0 and len(alt_vals) > 0:
            try:
                _, pval = sp_stats.mannwhitneyu(ref_vals, alt_vals, alternative="two-sided")
            except ValueError:
                pval = np.nan
            rb = rank_biserial(ref_vals, alt_vals)
            ref_med, alt_med = float(np.nanmedian(ref_vals)), float(np.nanmedian(alt_vals))
        else:
            pval, rb, ref_med, alt_med = np.nan, np.nan, np.nan, np.nan
        rows.append({
            "metric": metric_col, "comparison": f"{ref} vs {alt}",
            "n_ref": len(ref_vals), "n_alt": len(alt_vals),
            "median_ref": ref_med, "median_alt": alt_med,
            "p_value_raw": pval, "rank_biserial": rb,
        })
    adj = holm_bonferroni([r["p_value_raw"] for r in rows])
    for r, a in zip(rows, adj):
        r["p_value_holm"] = a
        r["significant_alpha_0.05"] = a < ALPHA
    return pd.DataFrame(rows)


stat_tables = {}
for name, df_ in [("expA_rf", expA_rf), ("expA_gbm", expA_gbm)]:
    for metric in ["shap_med_rho", "shap_med_jacc", "lime_med_rho", "lime_med_jacc"]:
        t = stratum_tests(df_, metric); t["source"] = name
        stat_tables[f"{name}_{metric}"] = t
for name, df_ in [("expB_rf", expB_rf), ("expB_gbm", expB_gbm)]:
    for metric in ["shap_lime_rho", "shap_lime_jacc"]:
        t = stratum_tests(df_, metric); t["source"] = name
        stat_tables[f"{name}_{metric}"] = t

all_stats = pd.concat(stat_tables.values(), ignore_index=True)
all_stats.to_csv(TABLES_DIR / "stat_tests_all.csv", index=False)
print(all_stats[["source","metric","comparison","median_ref","median_alt",
                 "p_value_holm","rank_biserial","significant_alpha_0.05"]].round(4).to_string(index=False))


  source         metric                               comparison  median_ref  median_alt  p_value_holm  rank_biserial  significant_alpha_0.05
 expA_rf   shap_med_rho          confident_correct vs borderline      0.9620      0.9180        0.0000        -0.4183                    True
 expA_rf   shap_med_rho confident_correct vs confident_incorrect      0.9620      0.9526        0.0000        -0.0893                    True
 expA_rf  shap_med_jacc          confident_correct vs borderline      1.0000      0.6667        0.0000        -0.3660                    True
 expA_rf  shap_med_jacc confident_correct vs confident_incorrect      1.0000      1.0000        0.0000         0.0100                    True
 expA_rf   lime_med_rho          confident_correct vs borderline      0.9713      0.9713        0.0000         0.0195                    True
 expA_rf   lime_med_rho confident_correct vs confident_incorrect      0.9713      0.9694        0.0000        -0.1380                    True
 expA_

## 14 · Summary figures

Side-by-side stratum medians for RF vs GBM, and global feature importance (mean |SHAP value|) across the test set.

In [17]:
def figure_summary_per_stratum(expA_rf, expA_gbm, metric_col, ylabel, title_suffix, fname):
    order = ["confident_correct", "borderline", "confident_incorrect"]
    rf_ = expA_rf.groupby("stratum")[metric_col].median().reindex(order)
    gb_ = expA_gbm.groupby("stratum")[metric_col].median().reindex(order)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    x = np.arange(len(order)); w = 0.35
    ax.bar(x - w/2, rf_.values, width=w, label="Random Forest", color="#3776BD")
    ax.bar(x + w/2, gb_.values, width=w, label="Gradient Boosting", color="#D88A0E")
    ax.set_xticks(x); ax.set_xticklabels([s.replace("_", "-") for s in order])
    ax.set_ylabel(ylabel); ax.set_title(f"Stratum medians — {title_suffix}")
    ax.legend(); ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    out = FIGURES_DIR / fname
    fig.savefig(out, dpi=140, bbox_inches="tight"); plt.close(fig)
    print(f"Wrote {out}")


figure_summary_per_stratum(expA_rf, expA_gbm, "shap_med_rho",
                           "Median Spearman ρ", "TreeSHAP perturbation stability",
                           "summary_expA_shap_rho.png")
figure_summary_per_stratum(expA_rf, expA_gbm, "lime_med_rho",
                           "Median Spearman ρ", "LIME perturbation stability",
                           "summary_expA_lime_rho.png")
figure_summary_per_stratum(expB_rf, expB_gbm, "shap_lime_jacc",
                           "Median Top-5 Jaccard", "Cross-method (SHAP vs LIME)",
                           "summary_expB_jacc.png")


def figure_global_importance(models, X_test, top_n=10):
    import shap
    rows = {}
    for name, model in models.items():
        try:
            expl = shap.TreeExplainer(model)
            sv = expl(X_test.iloc[:1000])
            v = sv.values
            if v.ndim == 3:
                v = v[:, :, 1]
            mean_abs = np.abs(v).mean(axis=0)
        except Exception as e:
            log.warning(f"Global importance fallback for {name}: {e}")
            mean_abs = np.zeros(X_test.shape[1])
        rows[name] = pd.Series(mean_abs, index=X_test.columns)
    df = pd.DataFrame(rows).sort_values(list(rows.keys())[0], ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    df.plot.barh(ax=ax, color=["#3776BD", "#D88A0E"])
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("Global feature importance (top 10)")
    ax.invert_yaxis()
    fig.tight_layout()
    out = FIGURES_DIR / "global_importance.png"
    fig.savefig(out, dpi=140, bbox_inches="tight"); plt.close(fig)
    df.to_csv(TABLES_DIR / "global_importance.csv")
    print(f"Wrote {out}")
    return df


gi_df = figure_global_importance({"random_forest": rf, "gradient_boosting": gbm}, X_test)
print("\nTop-10 global features:")
print(gi_df.round(4))


Wrote outputs/figures/summary_expA_shap_rho.png
Wrote outputs/figures/summary_expA_lime_rho.png
Wrote outputs/figures/summary_expB_jacc.png
Wrote outputs/figures/global_importance.png

Top-10 global features:
                random_forest  gradient_boosting
RepayDelay_Sep         0.0810             0.4699
RepayDelay_Aug         0.0349             0.0706
Credit_Limit           0.0315             0.2234
RepayDelay_Jul         0.0283             0.0789
Payment_Sep            0.0198             0.1210
Payment_Aug            0.0195             0.1348
RepayDelay_Jun         0.0194             0.0458
Payment_Jul            0.0181             0.0922
RepayDelay_May         0.0151             0.0401
BillAmt_Sep            0.0146             0.1373


## 15 · Headline numbers for the Results section

Generates `outputs/tables/RESULTS_HEADLINE.md` containing every number you need to write the Results section.

In [18]:
def build_results_headline():
    rf_best = rf_grid.iloc[0].to_dict()
    gbm_best = gbm_grid.iloc[0].to_dict()
    order = ["confident_correct", "borderline", "confident_incorrect"]

    def _med_table(df, cols):
        return df.groupby("stratum")[cols].median().reindex(order).round(3)

    expA_rf_med  = _med_table(expA_rf,  ["shap_med_rho","shap_med_jacc","lime_med_rho","lime_med_jacc"])
    expA_gbm_med = _med_table(expA_gbm, ["shap_med_rho","shap_med_jacc","lime_med_rho","lime_med_jacc"])
    expB_rf_med  = _med_table(expB_rf,  ["shap_lime_rho","shap_lime_jacc"])
    expB_gbm_med = _med_table(expB_gbm, ["shap_lime_rho","shap_lime_jacc"])

    out = []
    out.append("# Results headline numbers (auto-generated)")
    out.append("")
    out.append("## 1. Validation AUC at hyperparameter selection")
    out.append(f"- Random Forest:     `val_auc_roc = {rf_best['val_auc_roc']:.4f}` "
               f"(n_estimators={rf_best['n_estimators']}, max_depth={rf_best['max_depth']}, "
               f"min_samples_leaf={rf_best['min_samples_leaf']})")
    out.append(f"- Gradient Boosting: `val_auc_roc = {gbm_best['val_auc_roc']:.4f}` "
               f"(n_estimators={gbm_best['n_estimators']}, max_depth={gbm_best['max_depth']}, "
               f"learning_rate={gbm_best['learning_rate']})")
    out.append("")
    out.append("## 2. Test-set performance"); out.append("```")
    out.append(test_metrics.round(4).to_string()); out.append("```\n")
    out.append("## 3. Stratum counts (test set)"); out.append("```")
    out.append(counts.to_string()); out.append("```\n")
    out.append("## 4. Experiment A — perturbation stability (per-instance medians by stratum)")
    out.append("### Random Forest"); out.append("```\n" + expA_rf_med.to_string() + "\n```")
    out.append("### Gradient Boosting"); out.append("```\n" + expA_gbm_med.to_string() + "\n```\n")
    out.append("## 5. Experiment B — cross-method agreement (SHAP vs LIME)")
    out.append("### Random Forest"); out.append("```\n" + expB_rf_med.to_string() + "\n```")
    out.append("### Gradient Boosting"); out.append("```\n" + expB_gbm_med.to_string() + "\n```\n")
    out.append("## 6. Statistical tests (Mann–Whitney U, Holm–Bonferroni)")
    out.append("Significant at α = 0.05 means the stratum differs from Confident-Correct.")
    out.append("```")
    out.append(all_stats[["source","metric","comparison","median_ref","median_alt",
                          "p_value_holm","rank_biserial","significant_alpha_0.05"]]
               .round(4).to_string(index=False))
    out.append("```\n")
    out.append("## 7. σ-sweep robustness (Random Forest only)")
    out.append("```\n" + sigma_df.round(3).to_string(index=False) + "\n```")
    return "\n".join(out)


headline = build_results_headline()
(TABLES_DIR / "RESULTS_HEADLINE.md").write_text(headline)
print(headline)


# Results headline numbers (auto-generated)

## 1. Validation AUC at hyperparameter selection
- Random Forest:     `val_auc_roc = 0.7831` (n_estimators=500, max_depth=nan, min_samples_leaf=10)
- Gradient Boosting: `val_auc_roc = 0.7817` (n_estimators=200, max_depth=4, learning_rate=0.05)

## 2. Test-set performance
```
                   random_forest  gradient_boosting
auc_roc                   0.7787             0.7778
auc_pr                    0.5609             0.5557
accuracy                  0.7927             0.7649
precision_default         0.5301             0.4754
recall_default            0.5487             0.6131
f1_default                0.5393             0.5356
balanced_accuracy         0.7053             0.7105
```

## 3. Stratum counts (test set)
```
                     random_forest  gradient_boosting
stratum                                              
confident_correct             1341                985
borderline                     575                971
confid

## 16 · Download outputs

In [19]:
import shutil
zip_path = shutil.make_archive("optA_outputs", "zip", root_dir=OUTPUT_DIR)
print(f"Wrote {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Run on Colab to auto-download. Otherwise grab the zip manually.")


Wrote /content/optA_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>